# 🌌 SparseChron: 4D Reconstruction with Your Own Custom Photos on Kaggle

This notebook lets you upload **10–15 burst photos from your phone** (e.g., pouring liquid, moving objects) and automatically:
1. Clones SparseChron, DUSt3R, and Depth Anything V2.
2. Extracts camera poses & metric depth priors from your custom photos.
3. Generates the 3D point cloud initialization.
4. Trains the **4D Gaussian Splatting engine** within 16GB T4 VRAM limits.
5. Renders and outputs a high-definition **MP4 video** of your dynamic 4D scene.

### ⚡ Step 0: Ensure GPU is Enabled
In Kaggle, on the right sidebar, ensure **Accelerator** is set to **GPU T4 x2** or **GPU T4**.

### 1. Install System Dependencies & Tools

In [ ]:
!apt-get update && apt-get install -y git ninja-build libglib2.0-0 libsm6 libxrender-dev libxext6 ffmpeg

### 2. Clone Repositories

In [ ]:
%cd /kaggle/working
!rm -rf SparseChron Depth-Anything-V2 dust3r data custom_raw_images outputs

!git clone https://github.com/rajhodedara/SparseChron.git
!git clone https://github.com/DepthAnything/Depth-Anything-V2.git
!git clone --recursive https://github.com/naver/dust3r

### 3. Install Python Dependencies

In [ ]:
%cd /kaggle/working/SparseChron
!pip install -r requirements.txt
!pip install einops trimesh opencv-python viser tyro

### 4. Provide Your Photos

Make sure your 10-15 images (.jpg or .png) are uploaded to /kaggle/working/custom_raw_images.
You can upload a zip file or individual photos directly via Kaggle file explorer.

In [ ]:
import os, shutil, glob
from pathlib import Path

raw_dir = Path("/kaggle/working/custom_raw_images")
raw_dir.mkdir(parents=True, exist_ok=True)

# If you uploaded a zip file to Kaggle, uncomment and set path to unzip:
# !unzip -q /kaggle/input/my-custom-photos/images.zip -d /kaggle/working/custom_raw_images

images = list(raw_dir.glob("*"))
print(f"Total images ready in {raw_dir}: {len(images)}")
if len(images) == 0:
    print("⚠️ Please upload your 10-15 photos into /kaggle/working/custom_raw_images before running Step 5!")

### 5. Preprocess Custom Images (DUSt3R Poses & Depth Anything V2)

In [ ]:
%cd /kaggle/working/SparseChron
import os
os.environ["PYTHONPATH"] = "/kaggle/working/SparseChron:/kaggle/working/Depth-Anything-V2:/kaggle/working/dust3r:/kaggle/working/dust3r/croco"

!python scripts/preprocess.py \
    --scene-dir /kaggle/working/custom_raw_images \
    --output-dir /kaggle/working/data \
    --device cuda

### 6. Generate Initial 3D Point Cloud

In [ ]:
%cd /kaggle/working/SparseChron
!python scripts/generate_init_ply.py --data-dir /kaggle/working/data

### 7. Train 4D Gaussian Splatting Model

In [ ]:
%cd /kaggle/working/SparseChron
!PYTHONPATH=/kaggle/working/SparseChron python scripts/train.py \
    --scene-dir /kaggle/working/data \
    --output-dir /kaggle/working/outputs/custom_exp1 \
    --num-iterations 10000 \
    --is-4d

### 8. Render Dynamic Frames from Best Checkpoint

In [ ]:
%cd /kaggle/working/SparseChron
import glob
checkpoints = sorted(glob.glob("/kaggle/working/outputs/custom_exp1/*.ckpt"))
if checkpoints:
    best_ckpt = checkpoints[-1]
    print(f"Rendering from checkpoint: {best_ckpt}")
    !PYTHONPATH=/kaggle/working/SparseChron python scripts/evaluate.py \
        --checkpoint-path {best_ckpt} \
        --dataset-path /kaggle/working/data \
        --output-dir /kaggle/working/SparseChron/evaluation_output
else:
    print("No checkpoint found in outputs folder.")

### 9. View Evaluation Metrics (PSNR, SSIM, LPIPS)

In [ ]:
%cd /kaggle/working/SparseChron
import json
try:
    with open("evaluation_output/metrics.json", "r") as f:
        metrics = json.load(f)
        print(json.dumps(metrics, indent=4))
except Exception as e:
    print(f"Could not load metrics: {e}")

### 10. Generate Final 4D MP4 Video

In [ ]:
%cd /kaggle/working/SparseChron
!ffmpeg -y -framerate 15 -i evaluation_output/renders/render_%05d.png -c:v libx264 -pix_fmt yuv420p /kaggle/working/my_4D_scene.mp4
print("🎉 Video generated successfully! Download /kaggle/working/my_4D_scene.mp4 from the Kaggle file explorer.")